In [2]:
!pip install polars

Looking in indexes: https://artifactory.s.o3.ru/artifactory/api/pypi/pypi-virtual/simple
  Using cached https://artifactory.s.o3.ru/artifactory/api/pypi/pypi-virtual/packages/packages/b4/db/08f4ca10c5018813e7e0b59e4472302328b3d2ab1512f5a2157a814540e0/polars-1.39.3-py3-none-any.whl (823 kB)
  Using cached https://artifactory.s.o3.ru/artifactory/api/pypi/pypi-virtual/packages/packages/b0/15/fc3e43f3fdf3f20b7dfb5abe871ab6162cf8fb4aeabf4cfad822d5dc4c79/polars_runtime_32-1.39.3-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (46.9 MB)


In [3]:
import polars as pl
import torch
from datetime import date, timedelta
from tqdm import tqdm

In [4]:
TEST_START = date(2024, 7, 1)

In [5]:
user_actions_full = pl.read_parquet('./data/data/user_actions_full')

In [6]:
train_orders = (
    user_actions_full
    .filter(pl.col('date') < TEST_START)
    .filter(pl.col('date') >= TEST_START - timedelta(days=3 * 30))
    .filter(pl.col('action_type') == 'order')
    .select('user_id', 'product_id', 'date')
)

In [58]:
train_orders.filter(pl.col('user_id') == 2569829).sort('date')

user_id,product_id,date
i32,i64,date
2569829,1331147032,2024-05-03
2569829,536509373,2024-05-03
2569829,711289810,2024-05-03
2569829,1331147032,2024-05-11
2569829,325434481,2024-05-11
2569829,1064996339,2024-05-11
2569829,1331146939,2024-05-11
2569829,25343359,2024-06-05
2569829,522991547,2024-06-05


In [56]:
train_orders.filter(pl.col('user_id') == 2569829)

user_id,product_id,date
i32,i64,date
2569829,25343359,2024-06-05
10498283,171803041,2024-06-02
8368519,933027806,2024-06-21
10691321,679778603,2024-06-27
2062459,263792927,2024-06-28
…,…,…
5219201,383982855,2024-06-10
10681062,149086710,2024-05-08
415136,145923184,2024-04-06


In [7]:
del user_actions_full

In [8]:
id_mapping = (
    train_orders
    .select('product_id')
    .unique()
    .sort('product_id')
    .with_row_index('id')
    .with_columns([
        (pl.col('id') + 1).alias('id')
    ])
)

In [9]:
id_mapping

id,product_id
u32,i64
1,2579413
2,3094337
3,4036758
4,4036759
5,4036767
…,…
53751,1617064750
53752,1617136377
53753,1617149230


In [10]:
user_with_ids = (
    train_orders
    .join(id_mapping, on='product_id')
    .sort('user_id', 'date', 'id')
    .group_by('user_id')
    .agg(pl.col('id').alias('ids'))
)

In [11]:
(
    user_with_ids
    .with_columns([
        pl.col('ids').list.len().alias('ids_len')
    ])
    .select(
        pl.max('ids_len').alias('max_len'),
        pl.min('ids_len').alias('min_len'),
        pl.quantile('ids_len', 0.5).alias('median_len'),
    )
)

max_len,min_len,median_len
u32,u32,f64
1752,1,5.0


In [12]:
user_with_ids_filtered = (
    user_with_ids
    .with_columns([
        pl.col('ids').list.slice(-64, 64).alias('ids'),
    ])
    .filter(pl.col('ids').list.len() > 1)
)
user_with_ids_filtered

user_id,ids
i32,list[u32]
24,"[30136, 39889]"
62,"[1525, 1585, … 20743]"
64,"[7082, 9663, … 29492]"
83,"[2010, 3231, … 18802]"
121,"[13625, 14973, … 18353]"
…,…
11184097,"[95, 4150, … 28462]"
11184121,"[2265, 2354, … 32340]"
11184151,"[4092, 5075, … 41548]"


In [13]:
(
    user_with_ids_filtered
    .with_columns([
        pl.col('ids').list.len().alias('ids_len')
    ])
    .select(
        pl.max('ids_len').alias('max_len'),
        pl.min('ids_len').alias('min_len'),
        pl.quantile('ids_len', 0.5).alias('median_len'),
    )
)

max_len,min_len,median_len
u32,u32,f64
64,2,8.0


In [14]:
user_with_ids_filtered.shape

(713166, 2)

In [15]:
id_mapping.shape

(53755, 2)

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [17]:
class SASRec(nn.Module):
    def __init__(
        self,
        num_items,
        embed_dim=64, 
        num_heads=2,
        num_layers=2, 
        max_seq_len=64,
        dropout=0.1
    ):
        super().__init__()
        self.num_items = num_items
        self.embed_dim = embed_dim
        
        self.item_emb = nn.Embedding(num_items + 1, embed_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_seq_len + 1, embed_dim)
        self.dropout = nn.Dropout(dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=4*embed_dim,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, seq, valid_mask=None):
        B, L = seq.shape
        pos_ids = torch.arange(L, device=seq.device).unsqueeze(0).expand(B, L)
        
        x = self.item_emb(seq) + self.pos_emb(pos_ids)
        x = self.dropout(x)
        
        causal_mask = torch.triu(torch.ones(L, L, device=seq.device), diagonal=1).bool()
        key_padding_mask = ~valid_mask if valid_mask is not None else None
        
        x = self.transformer(x, mask=causal_mask, src_key_padding_mask=key_padding_mask)
        return self.norm(x)

In [18]:
class SeqDataset(Dataset):
    def __init__(self, seqs, labels, max_seq_len):
        self.max_seq_len = max_seq_len
        self.seqs, self.labels, self.masks = [], [], []
        for s, l in zip(seqs, labels):
            s, l = torch.as_tensor(s, dtype=torch.long), torch.as_tensor(l, dtype=torch.long)
            ln = min(len(s), max_seq_len)
            sp = torch.full((max_seq_len,), 0, dtype=torch.long)
            lp = torch.full((max_seq_len,), 0, dtype=torch.long)
            m = torch.zeros(max_seq_len, dtype=torch.bool)
            sp[:ln], lp[:ln], m[:ln] = s[:ln], l[:ln], True
            self.seqs.append(sp)
            self.labels.append(lp)
            self.masks.append(m)
        self.seqs = torch.stack(self.seqs)
        self.labels = torch.stack(self.labels)
        self.masks = torch.stack(self.masks)

    def __len__(self):
        return len(self.seqs)
        
    def __getitem__(self, i):
        return self.seqs[i], self.labels[i], self.masks[i]

In [19]:
train_seqs, train_labels = [], []
test_inputs, test_labels = [], []
for items in user_with_ids_filtered['ids'].to_list():
    train_seqs.append(items[:-1])
    train_labels.append(items[1:])
    
    test_inputs.append(items[:-1])
    test_labels.append(items[-1])

In [23]:
MAX_SEQ_LEN = 64
BATCH_SIZE = 2048
NUM_NEGATIVES = 16
EPOCHS = 10
LR = 5e-4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_ITEMS = id_mapping.shape[0]

In [24]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [25]:
train_ds = SeqDataset(train_seqs, train_labels, MAX_SEQ_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

In [26]:
model = SASRec(
    num_items=NUM_ITEMS,
    embed_dim=64,
    num_heads=2,
    num_layers=2, 
    max_seq_len=MAX_SEQ_LEN,
    dropout=0.1
).to(device)

/opt/conda/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [27]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [28]:
for epoch in range(EPOCHS):
    model.train()
    epoch_loss, n_valid = 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    for seqs, labels, masks in pbar:
        seqs, labels, masks = seqs.to(device), labels.to(device), masks.to(device)
        optimizer.zero_grad()
        
        seq_emb = model(seqs, masks)
        
        B, L, _ = seq_emb.shape
        neg_items = torch.randint(1, NUM_ITEMS+1, (B, L, NUM_NEGATIVES), device=device)
        #TODO проверить что негатив не равен позитиву
            
        pos_emb = model.item_emb(labels)
        neg_emb = model.item_emb(neg_items)
        
        pos_logits = (seq_emb * pos_emb).sum(-1).unsqueeze(-1)
        neg_logits = torch.einsum('bld,blnd->bln', seq_emb, neg_emb)
        logits = torch.cat([pos_logits, neg_logits], dim=-1)
        
        targets = torch.cat([torch.ones_like(pos_logits), torch.zeros_like(neg_logits)], dim=-1)
        valid_mask_exp = masks.unsqueeze(-1).expand_as(logits)
        
        loss = criterion(logits[valid_mask_exp], targets[valid_mask_exp])
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss.item() * valid_mask_exp.sum().item()
        n_valid += valid_mask_exp.sum().item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    scheduler.step()
    avg_loss = epoch_loss / max(n_valid, 1)

    # if (epoch + 1) % 5 == 0:
    #     rec = recall_at_k(model, test_loader, k, device, NUM_ITEMS)
    #     print(f"[Epoch {epoch+1}] Loss: {avg_loss:.4f} | Recall@{k}: {rec}")
    # else:
    print(f"[Epoch {epoch+1}] Loss: {avg_loss:.4f} (eval skipped)")

[Epoch 1] Loss: 1.4616 (eval skipped)


[Epoch 2] Loss: 0.2726 (eval skipped)


[Epoch 3] Loss: 0.1570 (eval skipped)


[Epoch 4] Loss: 0.1424 (eval skipped)


[Epoch 5] Loss: 0.1367 (eval skipped)


[Epoch 6] Loss: 0.1311 (eval skipped)


[Epoch 7] Loss: 0.1270 (eval skipped)


[Epoch 8] Loss: 0.1250 (eval skipped)


[Epoch 9] Loss: 0.1242 (eval skipped)


[Epoch 10] Loss: 0.1239 (eval skipped)


# done
- item_id -> обучаемые вектора
- позиционное кодирование


# todo
- признаки пользователя (например география)
- другое позиционное кодирование
- учет других действий, мы брали только заказы
- учет корзин/сессий

In [ ]:
[10, 20, 30] - [0, 1, 2]
[10, 20, 30] - [61, 62, 63] ???

In [30]:
item_embs = model.item_emb.weight
item_embs

Parameter containing:
tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [-1.8055,  0.5685,  0.0759,  ...,  0.1010, -0.2795, -0.3159],
        [ 2.2500, -0.3498, -0.9634,  ...,  0.1520,  1.2461, -1.5054],
        ...,
        [-0.8667,  0.4645,  0.5690,  ..., -1.0013,  0.1095,  1.5434],
        [ 0.2750, -1.2687,  0.1061,  ...,  1.5508,  0.9146,  1.2863],
        [-0.1042, -1.3599, -0.3283,  ...,  0.6997,  1.9032, -0.6406]],
       device='cuda:0', requires_grad=True)

In [31]:
class TestDataset(Dataset):
    def __init__(self, input_seqs, true_next_items, max_seq_len):
        self.max_seq_len = max_seq_len
        self.seqs, self.targets = [], []
        for s, t in zip(input_seqs, true_next_items):
            s = torch.as_tensor(s, dtype=torch.long)
            ln = min(len(s), max_seq_len)
            sp = torch.full((max_seq_len,), 0, dtype=torch.long)
            sp[:ln] = s[:ln]
            self.seqs.append(sp)
            self.targets.append(torch.tensor(t, dtype=torch.long))
        self.seqs = torch.stack(self.seqs)
        self.targets = torch.stack(self.targets)

    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, i):
        return self.seqs[i], self.targets[i]

def compute_user_embeddings(model, test_loader, device):
    model.eval()
    user_embs = []
    with torch.no_grad():
        for seqs, _ in test_loader:
            seqs = seqs.to(device)
            valid_mask = (seqs != 0)
            seq_emb = model(seqs, valid_mask)
            
            last_pos = valid_mask.sum(dim=1) - 1
            last_pos = torch.clamp(last_pos, min=0)
            batch_idx = torch.arange(len(seqs), device=device)
            user_emb = seq_emb[batch_idx, last_pos]
            user_embs.append(user_emb.cpu())
    return torch.cat(user_embs, dim=0)


In [32]:
test_ds = TestDataset(test_inputs, test_labels, MAX_SEQ_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [33]:
user_embs = compute_user_embeddings(model, test_loader, device)

In [53]:
val, ind = torch.topk(F.normalize(user_embs.to(device)[:1000]) @ F.normalize(item_embs[1:]).T, k=20)

In [55]:
val

tensor([[0.0718, 0.0637, 0.0627,  ..., 0.0411, 0.0404, 0.0402],
        [0.0667, 0.0603, 0.0601,  ..., 0.0413, 0.0402, 0.0400],
        [0.0718, 0.0621, 0.0611,  ..., 0.0404, 0.0401, 0.0388],
        ...,
        [0.0748, 0.0615, 0.0564,  ..., 0.0386, 0.0386, 0.0383],
        [0.0737, 0.0597, 0.0580,  ..., 0.0407, 0.0395, 0.0392],
        [0.0756, 0.0647, 0.0549,  ..., 0.0422, 0.0420, 0.0407]],
       device='cuda:0', grad_fn=<TopkBackward0>)

In [50]:
id_mapping.filter(pl.col('id') == 14152)

id,product_id
u32,i64
14152,356289422


https://github.com/MobileTeleSystems/RecTools/blob/main/examples/tutorials/transformers_tutorial.ipynb

https://github.com/asash/gSASRec-pytorch/blob/main/README.md